# AgrMap 

This notebook uses the functions in BaseMap.py to generate a calibrated, geospatially explicit agricultural map for the year 2020. The workflow produces a point layer containing harvested crop areas. Each crop is further divided into four main farming systems, consistent with the GAEZ classification: high-irrigated, high-rainfed, low-irrigated, and low-rainfed.

Before running the notebook, ensure that all required input data listed in the README file are prepared and fill the User Settings cell.

### Import Packages

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import rasterio
import requests
import geopandas as gpd
from shapely.geometry import Point
from rasterio.features import rasterize
from pyproj import CRS
import time
notebook_start = time.perf_counter()


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from Map.BaseMap import Raster, Map



### User Settings

In [ ]:
Country_name = 'ZMB'
cropland_value = 10  # Value for cropland in the land cover raster


## Coordinate and projection systems
crs_WGS84 = CRS("EPSG:4326")          # For the analysis, the original WGS84 coordinate system is used. 
crs_proj = CRS("EPSG:32735")          # Please provide proper projection system based on the geographical location of the selected country. 
                                      # The projection system has to be consistent with the LandCover raster.
                                      # More info: http://epsg.io/ 

### Directories


In [ ]:
ROOT_DIR = os.path.abspath(os.curdir)
OUT_NODATA = 255
Data_dir = os.path.join(ROOT_DIR, 'Data')
Input_data = os.path.join(Data_dir, '00.InputData')

land_cover_raster = os.path.join(Input_data, f'{Country_name}_LandCover.tif')


### Step 1: Create Base Cropland Layer

In [ ]:
output_cropland_raster = Raster.reclassify_land_cover(
    Country_name = Country_name,
    input_raster=land_cover_raster,
    output_dir=None,
    cropland_value=cropland_value,
    out_nodata=OUT_NODATA
)

In [ ]:
#converting all data that are not cropland into NoData. 

nodata_cropland_raster = os.path.splitext(output_cropland_raster)[0] + "_nod.tif"

with rasterio.open(output_cropland_raster) as src:
        profile = src.profile.copy()
        data = src.read(1)
        data = np.where(data == 0, OUT_NODATA, data)
        profile.update(nodata=OUT_NODATA)

with rasterio.open(nodata_cropland_raster, 'w', **profile) as dst:
        dst.write(data, 1)

In [ ]:
# #Run if the cell below returns an error. This is to check the fiona version and path.
# #If it returns "None" then run conda install -c conda-forge fiona geopandas

# import fiona
# print(fiona.__file__)
# print(getattr(fiona, "__version__", None))

In [ ]:
# Convert the cropland raster to points vector with coordinates
output_points_dir = Raster.raster_to_points(
    Country_name = Country_name,
    input_tif=None,
    output_dir=None,
    value_column='cropland'
)

### Step 2: Add Farmland Information

Rasterize farmland polygons to the same grid as the reference land cover raster,then sample those values on an existing GeoPackage and add a "farms" column.

Missing/no-farm pixels will have "NoFarm" in the 'farms' column.

In [ ]:
farmland_raster, farmland_gpkg = Raster.rasterize_farmlands(
    Country_name=Country_name,
    reference_raster=land_cover_raster,
    farmland_shapefile=os.path.join(Input_data, f"{Country_name}_farmlands.shp"),
    class_lookup_csv=os.path.join(Input_data, f"{Country_name}Farmlands.csv"),
    output_dir=None,
    out_nodata=OUT_NODATA,
)

### Step 3 – Process MAPSPAM Crop Rasters

Step 3 clips MAPSPAM rasters based on country boundaries and reproject them to the user defined coordinate system

In [ ]:
spam_input_dir = os.path.join(Input_data, "spam2020V2r2_global_harvested_area")
country_boundary = os.path.join(Input_data, f"gadm41_{Country_name}_0.shp")

spam_clip_dir, clipped_files = Raster.clip_rasters(
    Country_name=Country_name,
    input_dir=spam_input_dir,
    boundary_path=country_boundary,
)

print("Clipped rasters:", spam_clip_dir)

In [ ]:
spam_clip_dir = os.path.join(Data_dir, '02.ZMB_MAPSPAM_Crops', "01.ZMB_Clipped_Rasters" )
spam_reproj_dir, reprojected_files = Raster.reproject_rasters(
    Country_name=Country_name,
    clipped_dir=spam_clip_dir,
    out_nodata=OUT_NODATA,
    target_crs=crs_proj, 
)

print("Reprojected rasters:", spam_reproj_dir)

### Step 4 – Calculate the share of irrigated area and remove empty rasters 

In [ ]:
spam_reproj_dir = os.path.join(Data_dir, '02.ZMB_MAPSPAM_Crops', "02.Reprojected_Rasters" )
spam_filtered_dir, spam_summary_csv, spam_summary_df = Raster.mapspam_filter_crops(
    Country_name=Country_name,
    reprojected_dir=spam_reproj_dir,
    out_nodata=OUT_NODATA,
)

print("Filtered rasters:", spam_filtered_dir)
print("Summary CSV:", spam_summary_csv)

### Step 5 – Generate Density Rasters

In [ ]:
spam_filtered_dir =  os.path.join(Data_dir, '02.ZMB_MAPSPAM_Crops', "03.ZMB_Filtered_Rasters" )
spam_density_dir, density_files = Raster.calculate_density_rasters(
    Country_name=Country_name,
    filtered_dir=spam_filtered_dir,
    out_nodata=OUT_NODATA,
)

print("Density rasters:", spam_density_dir)
print(f"Created {len(density_files)} density rasters")

### Step 6 – Resample the rasters to the cropland point layer.

In [ ]:
#creates a GeoPackage with point geometries corresponding to the cropland locations
spam_density_dir = os.path.join(Data_dir, '02.ZMB_MAPSPAM_Crops', "04.Density_Rasters" )
spam_final_dir, spam_final_fp, spam_final_gdf = Map.resample_raster_to_points(
    Country_name=Country_name,
    input_dir=spam_density_dir,
    cropland_gpkg=os.path.join(Data_dir, "01.BaseCroplandLayer", f"{Country_name}_crp_farmland.gpkg"),
)

print("Final resampled layer:", spam_final_fp)
print("Saved to folder:", spam_final_dir)

In [ ]:
#This steps checks if there are any rows with a density higher than 1, which means that the harvested area is grater than the total pixel area. 

spam_final_gdf = os.path.join(
        Data_dir,
        "03.Final",
        f"01.{Country_name}_crop_IR",
        f"{Country_name}_crop_IR.gpkg",
    )
Pixel_sum= Map.calculate_pixel_sum(Country_name=Country_name, input_gpkg=spam_final_gdf)
print(Pixel_sum)

### Step 7 – Calibrate cropland

In [ ]:
#Calculate the total harvested area for each crop by summing the harvested area (calculated as density*pixel area) values across coordinate points in the GeoPackage.
spam_final_gdf = os.path.join(
        Data_dir,
        "03.Final",
        f"01.{Country_name}_crop_IR",
        f"{Country_name}_crop_IR.gpkg",
    )
mapspam_summary_csv = Map.mapspam_total_harvested_area(Country_name=Country_name, input_gpkg=spam_final_gdf)

print ("MAPSPAM summary CSV:", mapspam_summary_csv)

In [ ]:
#Calculate the national harvested area for each crop. This function reads the CropNames.csv file to create consistency across MAPSPAM naming 
#and the naming convention used in the national statistics. The result is the total havested area (irrigated or rainfed) for each crop in the country. 

National_harvested_area = Map.calculate_national_statistics(Country_name=Country_name)
print("National harvested area per crop:", National_harvested_area)

In [ ]:
#If there is a mismatch between the total harvested area calculated from the GeoPackage and the national statistics, a scaling factor is calculated to adjust the values in the GeoPackage.
Scaling_factor = Map.calculate_scaling_factor(Country_name=Country_name)
print("Scaling factors for irrigated and rainfed crops:", Scaling_factor)

In [ ]:
#Here we create a gpk file that multiplies per each row the crop density values with the scaling factor.
calibrated_gpkg = Map.calibrate_crops(Country_name)
print("Calibrated GeoPackage:", calibrated_gpkg)

#We need to check if the sum of each row is > 1. Having a sum > 1 means that the total density (Total hatvested area/total available area) is grater than 1
#This has no phisical meaning. The .csv file returns all the rows with sum > 1. 
pixel_sum_result = Map.calculate_pixel_sum(Country_name, calibrated_gpkg)
print("Pixel sum result:", pixel_sum_result)

#### Re-calibrate crop density


In [ ]:
# Redistribute any overloaded crop densities so every point sum is <= 1.
calibrated2_gpkg = Map.redistribute_crop_density(
    Country_name=Country_name,
    target_crs=crs_proj,
)
print("Redistributed GeoPackage:", calibrated2_gpkg)

In [ ]:
# Validate the redistributed file: no row should exceed a total density of 1.
pixel_sum_check = Map.calculate_pixel_sum(Country_name, calibrated2_gpkg)
print("Pixel sum validation result:", pixel_sum_check)

In [ ]:
check_harvested_area = Map.mapspam_total_harvested_area(Country_name=Country_name, input_gpkg=calibrated2_gpkg)
print(check_harvested_area)

### Step 8 – Assign Farming Systems

This step reclassifies harvested area based on the four main farming systems: HI (high-irrigated), LI (low-irrigated), HR (high-rainfed), LR (low-rainfed). The value corresponds to the hectares (ha) of harvested area. Step 8 returns 1 cropland.gpkg whith all the crops and 1 irrigated.gpkg which extrapolates irrigated-only crops. Three CSV files with final statistics are created: commercial farming share, irrigation share, and final statistics that summarize for each crop the total harvested area in hectares on a national scale. All the data are stored in folder Data/03.Final.

In [ ]:
final_gpkg = Map.create_final_cropland_map(Country_name= Country_name)
print(final_gpkg)


In [ ]:
stats = Map.export_final_cropland_statistics(Country_name=  Country_name)
print(stats)

In [ ]:
# Create irrigated-only GeoPackage (keeps columns ending with 'I')
irrigated_gpkg = Map.create_irrigated_gpkg(Country_name=Country_name)
print('Irrigated GeoPackage:', irrigated_gpkg)

In [ ]:
import time
elapsed = time.perf_counter() - notebook_start
print(f"The workflow took: {elapsed:.2f} seconds")